In [26]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, SimpleRNN, Dense
from tensorflow.keras.models import Model

In [27]:
with open('nombres2.txt', 'r', encoding='utf-8') as f:
    nombres = f.read().lower()

In [28]:
alfabeto = sorted(list(set(nombres)))
tam_alfabeto = len(alfabeto)

In [31]:
char_to_ix = { car:ind for ind, car in enumerate(alfabeto) }
ix_to_char = { ind:car for ind, car in enumerate(alfabeto) }

print(f"Tamaño del alfabeto: {tam_alfabeto}")
print(f"Caracteres: {alfabeto}")

Tamaño del alfabeto: 25
Caracteres: ['\n', ' ', '-', 'a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 't', 'u', 'v', 'x', 'y', 'z']


In [32]:
# 2. Definición del Modelo RNN
n_a = 25    # Número de unidades en la capa oculta

In [33]:
entrada = Input(shape=(None, tam_alfabeto), name="Entrada_Caracteres")
a0 = Input(shape=(n_a,), name="Estado_Inicial")

capa_recurrente = SimpleRNN(n_a, activation='tanh', return_state=True, return_sequences=True)
capa_salida = Dense(tam_alfabeto, activation='softmax', name="Salida_Softmax")

hs, _ = capa_recurrente(entrada, initial_state=a0)
salida = capa_salida(hs) # Eliminamos la lista innecesaria

In [34]:
modelo = Model([entrada, a0], salida)
modelo.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])
modelo.summary()

lineas = nombres.strip().split('\n')
np.random.shuffle(lineas)


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Entrada_Caracteres  │ (None, None, 25)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Estado_Inicial      │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_4        │ [(None, None,     │      1,275 │ Entrada_Caracter… │
│ (SimpleRNN)         │ 25), (None, 25)]  │            │ Estado_Inicial[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Salida_Softmax      │ (None, None, 25)  │        650 │ simple_rnn_4[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,925 (7.52 KB)

 Trainable params: 1,925 (7.52 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
def generador_datos(lista_nombres, char_map, vocab_size):
    while True:
        for nombre in lista_nombres:
            if not nombre:
                continue
            # Añadimos un salto de línea al final para que aprenda cuándo terminar
            palabra = nombre + '\n'

            # X: caracteres de entrada (del 0 al penúltimo)
            # Y: caracteres de salida esperados (del 1 al último)
            X_chars = palabra[:-1]
            Y_chars = palabra[1:]

            seq_len = len(X_chars)

            # Inicializar tensores One-Hot
            X = np.zeros((1, seq_len, vocab_size))
            Y = np.zeros((1, seq_len, vocab_size))

            for t, char in enumerate(X_chars):
                X[0, t, char_map[char]] = 1.0
            for t, char in enumerate(Y_chars):
                Y[0, t, char_map[char]] = 1.0

            # Estado inicial a0 (inicializado en ceros)
            state_a0 = np.zeros((1, n_a))

            yield (X, state_a0), Y

In [36]:
gen = generador_datos(lineas, char_to_ix, tam_alfabeto)

In [37]:
# 4. Entrenamiento del Modelo
print("\nComenzando entrenamiento...")
# Entrenamos por unas cuantas iteraciones (steps) de prueba, por ejemplo 5, y luego se recomienda entrenar por al menos 50
modelo.fit(gen, steps_per_epoch=len(lineas), epochs=50)


Comenzando entrenamiento...
Epoch 1/50
217/217 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.1547 - loss: 2.8666
Epoch 2/50
217/217 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2291 - loss: 2.5339
Epoch 3/50
217/217 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.2698 - loss: 2.3851
Epoch 4/50
217/217 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3051 - loss: 2.2505
Epoch 5/50
217/217 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3439 - loss: 2.1359
Epoch 6/50
217/217 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3854 - loss: 2.0441
Epoch 7/50
217/217 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4025 - loss: 1.9713
Epoch 8/50
217/217 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4107 - loss: 1.9119
Epoch 9/50
217/217 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4191 - loss: 1.8620
Epoch 10/50
217/217 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4342 - loss: 1.8193
Epoch 11/50
217/217 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4454 - loss: 1.7826
Epoch 12/50
217/217 ━━━

In [38]:
def generar_nombre(modelo, char_to_ix, ix_to_char, tam_alfabeto, n_a, max_len=20, temperatura=1.0):
    """
    Genera un nombre letra por letra utilizando el modelo entrenado.
    La 'temperatura' controla la creatividad:
    - Valores bajos (ej. 0.2) hacen que sea muy conservador (letras más probables).
    - Valores altos (ej. 1.2) lo hacen más creativo y variado, pero puede inventar palabras extrañas.
    """
    # 1. Elegir una letra inicial al azar (excluyendo el salto de línea '\n')
    letras_iniciales = [c for c in char_to_ix.keys() if c != '\n']
    char_actual = np.random.choice(letras_iniciales)

    nombre_generado = char_actual
    state_a0 = np.zeros((1, n_a)) # Estado inicial de la RNN en ceros

    for _ in range(max_len):
        # 2. Vectorizar la palabra que llevamos generada hasta ahora
        seq_len = len(nombre_generado)
        X = np.zeros((1, seq_len, tam_alfabeto))
        for t, char in enumerate(nombre_generado):
            X[0, t, char_to_ix[char]] = 1.0

        # 3. Predecir probabilidades para el siguiente carácter
        # Usamos verbose=0 para que no imprima barras de progreso en la consola
        predicciones = modelo.predict([X, state_a0], verbose=0)

        # Tomamos la distribución de probabilidad de la última letra predicha (último paso de tiempo)
        probabilidades = predicciones[0, -1, :]

        # 4. Aplicar temperatura para ajustar la creatividad
        probabilidades = np.log(probabilidades + 1e-8) / temperatura
        exp_probs = np.exp(probabilidades)
        probabilidades = exp_probs / np.sum(exp_probs)

        # 5. Muestrear la siguiente letra según las nuevas probabilidades
        idx_siguiente = np.random.choice(range(tam_alfabeto), p=probabilidades)
        char_siguiente = ix_to_char[idx_siguiente]

        # Si genera el salto de línea, significa que la palabra ha terminado
        if char_siguiente == '\n':
            break

        nombre_generado += char_siguiente

    return nombre_generado

In [39]:
print("\n--- Nombres generados por la IA ---")
for i in range(10):
    # temperatura=0.8 es un buen equilibrio entre coherencia y creatividad
    nuevo_nombre = generar_nombre(modelo, char_to_ix, ix_to_char, tam_alfabeto, n_a, temperatura=0.8)
    print(f"{i+1}: {nuevo_nombre.capitalize()}")


--- Nombres generados por la IA ---
1: Tlahuaxochitl
2: Yayatol
3: Huitemocomotziqui
4: Yahuel
5: Yihuatzomatl
6: Palcacihtlochitz
7: Yahualcochillaocoztli
8: Yahuatzin
9: Dehuitezomaut
10: Vaeotlacatzin


In [40]:
def generador_por_batches(lista_nombres, char_map, vocab_size, batch_size=32, max_len=20):
    """
    Generador que agrupa los nombres en lotes (batches) de tamaño fijo.
    Los nombres más cortos se van a rellenar con espacios ' ' para que todos midan 'max_len'.
    """
    num_ejemplos = len(lista_nombres)

    while True:
        # Mezclar los nombres al inicio de cada época
        indices = np.arange(num_ejemplos)
        np.random.shuffle(indices)

        # Iterar sobre el dataset en bloques de tamaño 'batch_size'
        for i in range(0, num_ejemplos, batch_size):
            # Obtener los nombres del lote actual
            lote_indices = indices[i:i + batch_size]
            lote_nombres = [lista_nombres[idx] for idx in lote_indices]

            # Ajustar el tamaño del lote final (por si quedan menos de 'batch_size' nombres al final)
            actual_batch_size = len(lote_nombres)

            # Inicializar matrices con ceros -Tensores como arrays de NumPy
            # Entrada X (caracteres de 0 a max_len - 1)
            X = np.zeros((actual_batch_size, max_len, vocab_size))
            # Salida Y (caracteres recorridos un paso a la derecha, de 1 a max_len)
            Y = np.zeros((actual_batch_size, max_len, vocab_size))

            for n_idx, nombre in enumerate(lote_nombres):
                # Preparar la palabra completa: añadimos salto de línea y rellenamos con espacios
                palabra_completa = (nombre + '\n').ljust(max_len + 1, ' ')

                # Crear secuencias X (entrada) e Y (salida/objetivo)
                seq_X = palabra_completa[:-1]
                seq_Y = palabra_completa[1:]

                # Convertir a One-Hot
                for t in range(max_len):
                    char_x = seq_X[t]
                    char_y = seq_Y[t]

                    # Si el carácter no está en el mapa, usamos el espacio ' ' por defecto
                    idx_x = char_map.get(char_x, char_map.get(' ', 0))
                    idx_y = char_map.get(char_y, char_map.get(' ', 0))

                    X[n_idx, t, idx_x] = 1.0
                    Y[n_idx, t, idx_y] = 1.0

            # El estado inicial 'a0' debe coincidir con el tamaño actual del lote
            state_a0 = np.zeros((actual_batch_size, n_a), dtype=np.float32)

            # Generar una tupla como entrada
            yield (X, state_a0), Y

In [41]:
print(f"Re-construyendo el modelo con tam_alfabeto = {tam_alfabeto}...")

# Definición de entradas
entrada = Input(shape=(None, tam_alfabeto), name="Entrada_Caracteres")
a0 = Input(shape=(n_a,), name="Estado_Inicial")

# Capas
capa_recurrente = SimpleRNN(n_a, activation='tanh', return_state=True, return_sequences=True)
capa_salida = Dense(tam_alfabeto, activation='softmax', name="Salida_Softmax")

# Conexiones
hs, _ = capa_recurrente(entrada, initial_state=a0)
salida = capa_salida(hs)

# Compilar nuevo modelo compatible con 26 caracteres
modelo = Model([entrada, a0], salida)
modelo.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("¡Modelo reconstruido con éxito!")

Re-construyendo el modelo con tam_alfabeto = 25...
¡Modelo reconstruido con éxito!


In [42]:
# --- PASO 1: Completar el alfabeto PRIMERO ---
if ' ' not in alfabeto:
    alfabeto = sorted(alfabeto + [' '])

tam_alfabeto = len(alfabeto)
char_to_ix = { car:ind for ind, car in enumerate(alfabeto) }
ix_to_char = { ind:car for ind, car in enumerate(alfabeto) }

print(f"Alfabeto final ({tam_alfabeto} caracteres): {alfabeto}")

# --- PASO 2: Construir el modelo con el tam_alfabeto ya correcto ---
print(f"\nConstruyendo modelo con tam_alfabeto = {tam_alfabeto}...")

entrada        = Input(shape=(None, tam_alfabeto), name="Entrada_Caracteres")
a0             = Input(shape=(n_a,),               name="Estado_Inicial")

capa_recurrente = SimpleRNN(n_a, activation='tanh', return_state=True, return_sequences=True)
capa_salida     = Dense(tam_alfabeto, activation='softmax', name="Salida_Softmax")

hs, _  = capa_recurrente(entrada, initial_state=a0)
salida = capa_salida(hs)

modelo = Model([entrada, a0], salida)
modelo.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("¡Modelo construido!")

# --- PASO 3: Configurar el generador ---
batch_size      = 64
max_len_dataset = max(len(nombre) for nombre in lineas) + 1  # +1 por el '\n'
steps_per_epoch = int(np.ceil(len(lineas) / batch_size))

generador_entrenamiento = generador_por_batches(
    lineas,
    char_to_ix,
    tam_alfabeto,
    batch_size=batch_size,
    max_len=max_len_dataset
)

# --- PASO 4: Entrenar ---
modelo.fit(
    generador_entrenamiento,
    steps_per_epoch=steps_per_epoch,
    epochs=100
)

Alfabeto final (25 caracteres): ['\n', ' ', '-', 'a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 't', 'u', 'v', 'x', 'y', 'z']

Construyendo modelo con tam_alfabeto = 25...
¡Modelo construido!
Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.0146 - loss: 3.4740
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0183 - loss: 3.3124
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0250 - loss: 3.1648
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1675 - loss: 3.0281
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4709 - loss: 2.8906
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5714 - loss: 2.7427
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6150 - loss: 2.5791
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6367 - loss: 2.4070
Epoch 9/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6403 - loss: 2.2400
Epoch 10/100
4/4 ━━

In [43]:
print("\n--- Nombres generados por la IA ---")
for i in range(10):
    # temperatura=0.8 es un buen equilibrio entre coherencia y creatividad
    nuevo_nombre = generar_nombre(modelo, char_to_ix, ix_to_char, tam_alfabeto, n_a, temperatura=0.8)
    print(f"{i+1}: {nuevo_nombre.capitalize()}")


--- Nombres generados por la IA ---
1: R-xjxopzutlioaaazmzii
2: Inpuaahea
3: Aaami i              
4: Mx-ehtlittc
5:  on
6: -jh-tmhitlhlocanuclt
7: Epoauaizi  a         
8: Orlzilchaacptdcole ot
9: Ji
10: Grxnemuacrauatlc
